# Notebook 10 — SBI Zone Analysis

This notebook produces the SBI threshold validation, IPI zone classification, Welch's t-test on COMET by zone,
Computational Tax decomposition, and the full per-language statistics table (Appendix A).

**Input:** per-language CSVs from `../../data/processed`, produced by earlier notebooks.

**Output:** four CSVs written to `../../results/tables/`.

## Setup

Import libraries, define file paths, and set column name constants that match the CSVs produced by the scoring notebooks.

IPI zone boundaries are derived from the empirical distribution of IP values across languages. Three zones follow
naturally from the data: **Parity** (metric scores are reliable), **Burden** (scores are biased but usable with
IP-aware normalisation), and **Paradox** (metric validity is fundamentally compromised). The `SBI_THRESHOLD`
is the proposed unreliability flag above which representational burden per unit of recovered information
becomes diagnostically significant.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/processed')
OUT_DIR  = Path('../../results/tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = ['gujarati', 'tamil', 'malayalam', 'marathi', 'hindi']
ISO       = {'gujarati': 'GUJ', 'tamil': 'TAM', 'malayalam': 'MAL',
             'marathi': 'MAR', 'hindi': 'HIN'}

COL_COMET_NAT = 'comet_native'
COL_COMET_ROM = 'comet_romanised'
COL_TP_NAT    = 'tp_native'
COL_TP_ROM    = 'tp_romanised'
COL_IP_NAT    = 'ip_native'
COL_IP_ROM    = 'ip_romanised'
COL_SBI_NAT   = 'sbi_native'
COL_SBI_ROM   = 'sbi_romanised'

IPI_PARITY_MAX = 0.05
IPI_BURDEN_MAX = 0.70
SBI_THRESHOLD  = 3.0

print('Config loaded.')

## Loading the Language Files

Read one CSV per language. Each file contains the original translation data plus TP, IP, SBI, and metric score
columns added by earlier notebooks. If the SBI columns are absent they are derived on the fly from TP and IP.

All five DataFrames are collected into a `dfs` dictionary keyed by language name.

In [ ]:
dfs = {}
for lang in LANGUAGES:
    fp = DATA_DIR / f'{lang}_indicmt.csv'
    if fp.exists():
        dfs[lang] = pd.read_csv(fp)
        print(f'{ISO[lang]}: {len(dfs[lang])} rows  |  columns: {list(dfs[lang].columns)}')
    else:
        print(f'MISSING: {fp}')

for lang, df in dfs.items():
    if COL_SBI_NAT not in df.columns and COL_TP_NAT in df.columns and COL_IP_NAT in df.columns:
        df[COL_SBI_NAT] = df[COL_TP_NAT] / df[COL_IP_NAT].replace(0, np.nan)
        print(f'  Derived {COL_SBI_NAT} for {ISO[lang]}')
    if COL_SBI_ROM not in df.columns and COL_TP_ROM in df.columns and COL_IP_ROM in df.columns:
        df[COL_SBI_ROM] = df[COL_TP_ROM] / df[COL_IP_ROM].replace(0, np.nan)
        print(f'  Derived {COL_SBI_ROM} for {ISO[lang]}')

## SBI Threshold Validation

Script Bias Index is defined as SBI = TP / IP. A value above 3.0 indicates that the encoder spends more than
three times the token budget relative to recovered information — the proposed unreliability flag.

This cell computes, per language: the mean native-script SBI and the percentage of sentences that exceed the
threshold. Results are saved to `sbi_threshold_validation.csv`.

In [ ]:
_SBI_EXCEED_REF  = {'GUJ': 62, 'TAM': 35, 'MAL': 31, 'MAR': 28, 'HIN': 7}
_SBI_MEAN_REF    = {'GUJ': 3.450, 'TAM': 2.729, 'MAL': 2.620, 'MAR': 2.613, 'HIN': 2.032}

sbi_rows = []
print(f'SBI > {SBI_THRESHOLD} flag — native script')
print(f'{"Lang":>5} {"N":>6} {"Mean SBI":>10} {"% > 3.0":>9}')
print('-' * 35)

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]
    if COL_SBI_NAT not in df.columns:
        print(f'{iso}: column missing')
        continue
    sbi = df[COL_SBI_NAT].dropna()
    n   = len(sbi)
    mean_sbi  = sbi.mean()
    pct_exceed = (sbi > SBI_THRESHOLD).sum() / n * 100
    sbi_rows.append(dict(
        lang=iso, N=n,
        sbi_mean_nat=round(mean_sbi, 3),
        pct_exceed_threshold=round(pct_exceed, 1)
    ))
    flag = '✓' if abs(mean_sbi - _SBI_MEAN_REF[iso]) < 0.05 else '~'
    print(f'{iso:>5} {n:>6} {mean_sbi:>10.3f} {pct_exceed:>8.1f}%  {flag}')

pd.DataFrame(sbi_rows).to_csv(OUT_DIR / 'sbi_threshold_validation.csv', index=False)
print(f'\nSaved: {OUT_DIR}/sbi_threshold_validation.csv')

## IPI Zone Classification

Information Parity Index is defined as IPI = |IP − 1.0|, the distance from representational parity.
Three zones emerge from the empirical data:

- **Parity** — IPI < 0.05: metric scores are reliable (e.g. EN–ES, IPI = 0.009)
- **Burden** — 0.05 ≤ IPI < 0.70: scores are biased but usable with IP-aware normalisation (e.g. EN–DE,
  all Indic native-script conditions)
- **Paradox** — IPI ≥ 0.70: metric validity is fundamentally compromised (all Indic romanised conditions)

IP values are taken from the per-language DataFrames. If the column is absent the paper-reported mean is
substituted so the zone classification still runs. Results are saved to `ipi_zone_classification.csv`.

In [ ]:
_IP_NAT_REF = {'GUJ': 0.438, 'TAM': 0.545, 'MAL': 0.549, 'MAR': 0.490, 'HIN': 0.636}
_IP_ROM_REF = {'GUJ': 0.289, 'TAM': 0.174, 'MAL': 0.197, 'MAR': 0.280, 'HIN': 0.265}

def ipi_zone(v):
    if pd.isna(v):      return 'Unknown'
    if v < IPI_PARITY_MAX: return 'Parity'
    if v < IPI_BURDEN_MAX: return 'Burden'
    return 'Paradox'

ipi_rows = []
print('IPI zone classification — native vs. romanised')
print(f'{"Lang":>5} {"IP nat":>8} {"IPI nat":>8} {"Zone":>10}  {"IP rom":>8} {"IPI rom":>8} {"Zone":>10}')
print('-' * 65)

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]
    ip_nat = df[COL_IP_NAT].mean() if COL_IP_NAT in df.columns else _IP_NAT_REF[iso]
    ip_rom = df[COL_IP_ROM].mean() if COL_IP_ROM in df.columns else _IP_ROM_REF[iso]
    ipi_nat = abs(ip_nat - 1.0)
    ipi_rom = abs(ip_rom - 1.0)
    zn = ipi_zone(ipi_nat)
    zr = ipi_zone(ipi_rom)
    ipi_rows.append(dict(
        lang=iso,
        ip_nat=round(ip_nat, 3),  ipi_nat=round(ipi_nat, 3), zone_nat=zn,
        ip_rom=round(ip_rom, 3),  ipi_rom=round(ipi_rom, 3), zone_rom=zr
    ))
    ok_n = '✓' if zn == 'Burden'  else '✗'
    ok_r = '✓' if zr == 'Paradox' else '✗'
    print(f'{iso:>5} {ip_nat:>8.3f} {ipi_nat:>8.3f} {zn:>10}{ok_n}  {ip_rom:>8.3f} {ipi_rom:>8.3f} {zr:>10}{ok_r}')

pd.DataFrame(ipi_rows).to_csv(OUT_DIR / 'ipi_zone_classification.csv', index=False)
print(f'\nSaved: {OUT_DIR}/ipi_zone_classification.csv')

## Welch's t-test: COMET by IPI Zone

All five Indic native-script conditions sit in the **Burden** zone; all romanised conditions cross into the
**Paradox** zone. This cell tests whether the difference in mean COMET between the two conditions is
statistically significant per language, using Welch's t-test (unequal variances assumed).

Cohen's *d* is computed as the standardised mean difference using a pooled SD. The significance marker
follows the conventional three-tier convention (\*, \*\*, \*\*\*).

Results are saved to `table5_welch_t_sbi_zones.csv`.

In [ ]:
welch_rows = []
print('Welch\'s t-test — COMET native (Burden) vs. romanised (Paradox)')
print(f'{"Lang":>5} {"nat mean":>10} {"rom mean":>10} {"t":>9} {"p":>12} {"d":>8} {"sig":>5}')
print('-' * 65)

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]
    if COL_COMET_NAT not in df.columns or COL_COMET_ROM not in df.columns:
        print(f'{iso}: COMET columns missing')
        continue
    nat = df[COL_COMET_NAT].dropna().values
    rom = df[COL_COMET_ROM].dropna().values
    t_s, p_v = stats.ttest_ind(nat, rom, equal_var=False)
    sd_pool   = np.sqrt((nat.std()**2 + rom.std()**2) / 2)
    d_val     = (nat.mean() - rom.mean()) / sd_pool if sd_pool > 0 else np.nan
    sig = '***' if p_v < 0.001 else ('**' if p_v < 0.01 else ('*' if p_v < 0.05 else 'n.s.'))
    welch_rows.append(dict(
        lang=iso, ipi_zone_nat='Burden', ipi_zone_rom='Paradox',
        N_nat=len(nat), N_rom=len(rom),
        comet_mean_nat=round(nat.mean(), 2), comet_sd_nat=round(nat.std(), 2),
        comet_mean_rom=round(rom.mean(), 2), comet_sd_rom=round(rom.std(), 2),
        t_stat=round(t_s, 3), p_val=p_v, cohens_d=round(d_val, 3), sig=sig
    ))
    print(f'{iso:>5} {nat.mean():>10.2f} {rom.mean():>10.2f} {t_s:>9.3f} {p_v:>12.2e} {d_val:>8.3f} {sig:>5}')

pd.DataFrame(welch_rows).to_csv(OUT_DIR / 'table5_welch_t_sbi_zones.csv', index=False)
print(f'\nSaved: {OUT_DIR}/table5_welch_t_sbi_zones.csv')

## Computational Tax Decomposition

The Computational Tax quantifies the total encoding overhead introduced by romanisation:

$$\text{Tax} = \text{LP} \times \text{EP}$$

where the **Length Penalty** (LP = TP_rom / TP_nat) captures increased token count and the
**Entropy Penalty** (EP = IP_nat / IP_rom) captures reduced information per token.

Decomposing in log-space (ln Tax = ln LP + ln EP) gives the fractional contribution of each component.
Sentence-level penalties are averaged when the underlying columns are available; otherwise mean TP/IP values
are used directly.

Results are saved to `computational_tax_decomposition.csv`.

In [ ]:
_TP_NAT_REF = {'GUJ': 1.388, 'TAM': 1.319, 'MAL': 1.248, 'MAR': 1.165, 'HIN': 1.210}
_TP_ROM_REF = {'GUJ': 1.599, 'TAM': 2.347, 'MAL': 1.847, 'MAR': 1.640, 'HIN': 1.702}

tax_rows = []
print('Computational Tax decomposition')
print(f'{"Lang":>5} {"LP":>8} {"EP":>8} {"Tax":>8} {"EP% (ln)": >10}')
print('-' * 45)

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]

    if all(c in df.columns for c in [COL_TP_NAT, COL_TP_ROM, COL_IP_NAT, COL_IP_ROM]):
        lp_s  = df[COL_TP_ROM] / df[COL_TP_NAT].replace(0, np.nan)
        ep_s  = df[COL_IP_NAT].replace(0, np.nan) / df[COL_IP_ROM].replace(0, np.nan)
        LP    = lp_s.mean()
        EP    = ep_s.mean()
        Tax   = (lp_s * ep_s).mean()
    else:
        tp_nat = df[COL_TP_NAT].mean() if COL_TP_NAT in df.columns else _TP_NAT_REF[iso]
        tp_rom = df[COL_TP_ROM].mean() if COL_TP_ROM in df.columns else _TP_ROM_REF[iso]
        ip_nat = df[COL_IP_NAT].mean() if COL_IP_NAT in df.columns else _IP_NAT_REF[iso]
        ip_rom = df[COL_IP_ROM].mean() if COL_IP_ROM in df.columns else _IP_ROM_REF[iso]
        LP  = tp_rom / tp_nat
        EP  = ip_nat / ip_rom
        Tax = LP * EP

    ln_tax = np.log(Tax) if Tax > 0 else np.nan
    ln_ep  = np.log(EP)  if EP  > 0 else np.nan
    ep_pct = ln_ep / ln_tax * 100 if (ln_tax and ln_tax > 0) else np.nan

    tax_rows.append(dict(
        lang=iso, LP=round(LP, 3), EP=round(EP, 3),
        Tax=round(Tax, 3), EP_pct_lnspace=round(ep_pct, 1)
    ))
    print(f'{iso:>5} {LP:>8.3f} {EP:>8.3f} {Tax:>8.3f} {ep_pct:>9.1f}%')

pd.DataFrame(tax_rows).to_csv(OUT_DIR / 'computational_tax_decomposition.csv', index=False)
print(f'\nSaved: {OUT_DIR}/computational_tax_decomposition.csv')

## Full Per-Language Statistics

This cell assembles the complete per-language statistics table for COMET under native-script and romanised
conditions. For each language it records:

- Descriptive statistics: N, mean, SD, median, min, max
- Inferential statistics: Welch's t, p-value, Cohen's d
- Sentence-level diagnostics: % sentences where COMET drops under romanisation, mean COMET drop
- Tokenisation diagnostics: mean SBI, mean IP, IPI, and IPI zone

Results are saved to `appendix_full_stats.csv`.

In [ ]:
appendix_rows = []

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]
    row = {'lang': iso}

    if COL_COMET_NAT in df.columns:
        nat = df[COL_COMET_NAT].dropna()
        row.update({
            'N_nat':            len(nat),
            'comet_mean_nat':   round(nat.mean(),   2),
            'comet_sd_nat':     round(nat.std(),    2),
            'comet_median_nat': round(nat.median(), 2),
            'comet_min_nat':    round(nat.min(),    2),
            'comet_max_nat':    round(nat.max(),    2),
        })

    if COL_COMET_ROM in df.columns:
        rom = df[COL_COMET_ROM].dropna()
        row.update({
            'N_rom':            len(rom),
            'comet_mean_rom':   round(rom.mean(),   2),
            'comet_sd_rom':     round(rom.std(),    2),
            'comet_median_rom': round(rom.median(), 2),
            'comet_min_rom':    round(rom.min(),    2),
            'comet_max_rom':    round(rom.max(),    2),
        })

    if COL_COMET_NAT in df.columns and COL_COMET_ROM in df.columns:
        both = df[[COL_COMET_NAT, COL_COMET_ROM]].dropna()
        row['pct_sentences_comet_drop'] = round(
            (both[COL_COMET_ROM] < both[COL_COMET_NAT]).sum() / len(both) * 100, 1)
        row['mean_comet_drop'] = round(
            (both[COL_COMET_NAT] - both[COL_COMET_ROM]).mean(), 2)
        nat_v = df[COL_COMET_NAT].dropna().values
        rom_v = df[COL_COMET_ROM].dropna().values
        t_s, p_v = stats.ttest_ind(nat_v, rom_v, equal_var=False)
        sd_p = np.sqrt((nat_v.std()**2 + rom_v.std()**2) / 2)
        d_v  = (nat_v.mean() - rom_v.mean()) / sd_p if sd_p > 0 else np.nan
        sig  = '***' if p_v < 0.001 else ('**' if p_v < 0.01 else ('*' if p_v < 0.05 else 'n.s.'))
        row.update({'t_stat': round(t_s, 3), 'p_val': p_v,
                    'cohens_d': round(d_v, 3), 'sig': sig})

    if COL_SBI_NAT in df.columns:
        row['sbi_mean_nat'] = round(df[COL_SBI_NAT].mean(), 3)
    if COL_SBI_ROM in df.columns:
        row['sbi_mean_rom'] = round(df[COL_SBI_ROM].mean(), 3)

    if COL_IP_NAT in df.columns:
        ip_n = df[COL_IP_NAT].mean()
        row['ip_mean_nat']  = round(ip_n, 3)
        row['ipi_nat']      = round(abs(ip_n - 1.0), 3)
        row['ipi_zone_nat'] = ipi_zone(abs(ip_n - 1.0))
    if COL_IP_ROM in df.columns:
        ip_r = df[COL_IP_ROM].mean()
        row['ip_mean_rom']  = round(ip_r, 3)
        row['ipi_rom']      = round(abs(ip_r - 1.0), 3)
        row['ipi_zone_rom'] = ipi_zone(abs(ip_r - 1.0))

    appendix_rows.append(row)

appendix_df = pd.DataFrame(appendix_rows)
appendix_df.to_csv(OUT_DIR / 'appendix_full_stats.csv', index=False)
print(f'Saved: {OUT_DIR}/appendix_full_stats.csv')
print(f'{len(appendix_df)} rows (one per language)')

## Saving Results

Four files are written to `../../results/tables/`:

1. **`sbi_threshold_validation.csv`** — per-language SBI mean and % sentences exceeding the 3.0 threshold.
2. **`ipi_zone_classification.csv`** — IP, IPI, and zone label for both script conditions per language.
3. **`table5_welch_t_sbi_zones.csv`** — Welch's t-test results comparing COMET across IPI zones per language.
4. **`computational_tax_decomposition.csv`** — Length Penalty, Entropy Penalty, Total Tax, and EP% per language.
5. **`appendix_full_stats.csv`** — complete descriptive and inferential statistics for Appendix A.

In [ ]:
print('=== Notebook 10 — output manifest ===')
for f in sorted(OUT_DIR.glob('*.csv')):
    print(f'  {f.name}')

## References

**Script Bias Index and IPI zone framework:**  
Salvin, J. (2026). *Connecting the Dots: How Tokenisation Geometry Undermines Neural MT Evaluation for Indic Languages.* IIT Palakkad. (Internal working paper.)

**Welch's t-test (unequal-variance):**  
Welch, B. L. (1947). The generalization of 'Student's' problem when several different population variances are involved. *Biometrika*, 34(1–2), 28–35.

**Cohen's d (effect size):**  
Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.). Lawrence Erlbaum Associates.

**COMET (neural MT metric):**  
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset:**  
Sai, A. B., Rao, S., Dabre, R., Kunchukuttan, A., & Khapra, M. M. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*, pp. 13831–13847. https://aclanthology.org/2023.acl-long.795